# Ego network — u/CaseA

Directed ego ↔ alter reply network built from the
[Arctic Shift](https://arctic-shift.photon-reddit.com) API, to find the accounts the ego is in
contact with most often.

**Alters** — accounts the ego replied to directly, and accounts that replied to the ego
directly, on a comment (`parent_id = t1_…`) or a submission (`parent_id = t3_…`).

**Contacts** — the number of **distinct posts** (submissions) on which the ego and the alter
replied to each other directly, in either direction. A long comment chain on one post counts
once. Reply counts are kept as a secondary column.

**Removed**
- `[deleted]` / `[removed]` authors — not identifiable accounts.
- **bots** — accounts on the public Pushshift bot list (`ignored.txt`), `AutoModerator`, whose
  name ends in `bot` (not `robot`), or whose replies mostly identify themselves as a bot
  (section 6b). Accounts flagged only by the name rule are printed
  for review; add a false positive to `NOT_BOTS` to keep it.
- alters sharing fewer than `MIN_CONTACTS` (= 2) distinct posts with the ego.

Every account that Arctic Shift still has a name for is otherwise kept. If
`CHECK_ACCOUNT_STATUS` is on, each retained alter's *current* status is looked up with PRAW
and stored as an `account_status` attribute (`Active` / `Suspended` / `Deleted`); set
`DROP_PRAW_DELETED = True` to also remove accounts PRAW reports as deleted.

No alter–alter ties here — see the scoping notes for u/CaseA (≈25–35k alters, ≈250M
items of alter history).

---
## Collection strategy

The ego has ~16k comments and ~10k posts, so replies are **not** looked up comment by
comment. Every reply to the ego sits in a thread the ego posted or commented in, and so
does the parent of every ego comment. The notebook pulls **every archived comment in those
threads** (≈10.5k threads, ≈1.6M comments) and derives both directions from that one pull.

| API finding (verified live) | Consequence |
|---|---|
| the archive stores the name as `CaseA` (lower case) | `EGO` uses that spelling; author matches are case-insensitive anyway |
| `after` is exclusive | paging uses `after = last_created_utc - 1` and dedupes on `id` |
| `limit='auto'` returns 100–1000 rows | a page under 100 rows is the last page |
| `/ids` batches kept at 100 | avoids silent truncation |
| intermittent `422` | retried with backoff |

## 1. Imports and configuration

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests_ratelimiter import LimiterSession
from dotenv import load_dotenv
import pyarrow as pa
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

In [ ]:
# ---- configuration -------------------------------------------------------
EGO = 'CaseA'                 # spelling as stored in the archive

MIN_CONTACTS = 2                    # alters sharing fewer distinct posts with the ego are removed
NON_AUTHORS = {'[deleted]', '[removed]', None, ''}

# Bots: known list + AutoModerator + a name rule ('...bot', '...bot_2', not '...robot')
BOT_LIST_PATH = '/home/eeporter/redditdumps/PushshiftDumps/scripts/ignored.txt'
EXTRA_BOTS = {'AutoModerator', 'ExampleMirrorBot'}   # ExampleMirrorBot: mirror tables, verified by hand
NOT_BOTS = set()                    # names the name rule wrongly flags
BOT_NAME_RULE = re.compile(r'(?<!ro)(bot|b0t)[_-]?\d*$', re.IGNORECASE)
# reply-text rule (section 6b): self-identification or a bot footer linking to its code
BOT_TEXT_RULE = (r"(?i)\bi(?: am|'m|m) an? (?:bot|automated)\b|\^+i'?m \^+a \^+bot|performed automatically"
                 r"|self-deletes|beep,? boop|\[\^*\[?(?:code|source(?: code)?|github|faq|feedback|issues"
                 r"|about this bot|info)\]?\]\((?:https?://)?(?:www\.)?(?:github\.com|reddit\.com/r/\w*bot)")
BOT_TEXT_SHARE = 0.5                # share of an alter's replies that must match

CHECK_ACCOUNT_STATUS = True         # PRAW lookup of each retained alter (cached)
DROP_PRAW_DELETED = False           # True also removes accounts PRAW reports as Deleted
ENV_PATH = '/home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env'

N_WORKERS = 4                       # combined request rate stays at 5 req/s
REQUESTS_PER_SECOND = 5
THREAD_BATCH = 250                  # threads per cache file (resume granularity)
# CaseA comments in a handful of AskReddit/politics megathreads (up to 74k comments) where
# co-presence carries no information: the 511 threads hold 2.0M comments, 94% of them in 139 threads.
# Threads above the cap are dropped before the thread pull, so alter-alter ties are built only from
# threads small enough for shared presence to mean something.
THREAD_MAX_COMMENTS = 2000

COMMENT_FIELDS = ['id', 'author', 'parent_id', 'link_id', 'created_utc', 'subreddit']
POST_FIELDS    = ['id', 'author', 'created_utc', 'subreddit', 'num_comments']

TOP_N_GRAPH = 60                    # alters drawn in the network figure
TOP_N_BARS  = 30                    # alters in the ranked bar chart

OUT_DIR   = f'ego_net_{EGO}_output'
CACHE_DIR = os.path.join(OUT_DIR, 'thread_cache')
FIG_DIR   = os.path.join('..', 'p2_figures')
for d in (OUT_DIR, CACHE_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print(f'ego = {EGO}  min contacts = {MIN_CONTACTS}\noutputs -> {OUT_DIR}\nfigures -> {FIG_DIR}')

## 2. API layer

In [ ]:
# Arctic Shift API helpers, safe to call from several threads at once.
BASE = 'https://arctic-shift.photon-reddit.com/api'
# One rate-limited HTTP session per worker thread; the total rate is shared across N_WORKERS.
_local = threading.local()


def get_session():
    if not hasattr(_local, 'session'):
        _local.session = LimiterSession(per_minute=int(60 * REQUESTS_PER_SECOND / N_WORKERS))
        _local.session.headers['User-Agent'] = 'dissertation-ego-network/1.0 (research)'
    return _local.session


class BadQuery(Exception):
    """HTTP 400 - a malformed query that retrying cannot fix."""


def arctic_get(path, **params):
    """One GET with the standard retry guard. Raises after 10 failed attempts."""
    retry_count = 0
    last_error = None
    while retry_count < 10:
        try:
            response = get_session().get(f'{BASE}/{path}', params=params, timeout=120)
            if response.status_code == 200:
                return response.json().get('data') or []
            if response.status_code == 400:
                raise BadQuery(f'400 from {path} {params}: {response.text[:200]}')
            last_error = f'HTTP {response.status_code}'
        except BadQuery:
            raise
        except Exception as e:
            last_error = f'{e.__class__.__name__}: {e}'
        retry_count += 1
        print(f'  retry {retry_count} for {path} {params}: {last_error}')
        time.sleep(min(2 ** retry_count, 60))
    raise RuntimeError(f'Failed {path} {params} after 10 retries. Last: {last_error}')


def arctic_paginate(path, limit='auto', **params):
    """Page a search endpoint in ascending created_utc order (`after` is exclusive, so the
    cursor sits one second before the last row seen; rows are deduplicated on id)."""
    rows, seen = [], set()
    after = None
    while True:
        page_params = dict(params, limit=limit, sort='asc')
        if after is not None:
            page_params['after'] = after
        data = arctic_get(path, **page_params)
        new = [r for r in data if r['id'] not in seen]
        seen.update(r['id'] for r in new)
        rows.extend(new)
        if len(data) < 100:
            break
        last = data[-1]['created_utc']
        if not new:
            print(f'  WARNING cursor stalled at {last} for {path} {params}; stepping past it')
            after = last
        else:
            after = last - 1
    return rows


# Look up posts or comments by id, 100 per request (the endpoint silently truncates above 100).
def fetch_by_ids(kind, ids, fields):
    ids = [i for i in dict.fromkeys(ids) if i]
    out = []
    for start in range(0, len(ids), 100):
        batch = ids[start:start + 100]
        data = arctic_get(f'{kind}/ids', ids=','.join(batch), fields=fields)
        assert len(data) <= len(batch)
        out.extend(data)
    return out


print('API layer ready')

## 3. The ego's own activity

In [ ]:
# Every comment and post the ego account made, with checks that the author really is the ego and ids are unique.
ego_comments = pd.DataFrame(arctic_paginate('comments/search', author=EGO,
                                            fields=','.join(COMMENT_FIELDS)), columns=COMMENT_FIELDS)
ego_posts = pd.DataFrame(arctic_paginate('posts/search', author=EGO,
                                         fields=','.join(POST_FIELDS)), columns=POST_FIELDS)

assert len(ego_comments) or len(ego_posts), f'no activity found for {EGO}'
assert (ego_comments['author'].str.lower() == EGO.lower()).all()
assert (ego_posts['author'].str.lower() == EGO.lower()).all()
assert ego_comments['id'].is_unique and ego_posts['id'].is_unique

# Activity counts, active period, and what the ego's comments reply to (t1_ = a comment, t3_ = a post).
_ts = pd.concat([ego_comments['created_utc'], ego_posts['created_utc']])
print(f'ego comments: {len(ego_comments):,}')
print(f'ego posts   : {len(ego_posts):,}')
print(f'active      : {pd.to_datetime(_ts.min(), unit="s")} -> {pd.to_datetime(_ts.max(), unit="s")}')
print(f'comment parents: {ego_comments["parent_id"].str[:3].value_counts().to_dict()}')

## 4. Pull every thread the ego took part in

Threads are fetched in parallel and cached in batches of `THREAD_BATCH`, so an interrupted
run resumes from the last complete batch.

In [ ]:
# Every thread the ego took part in: its own posts plus the threads of its comments.
own_links = ('t3_' + ego_posts['id']).tolist()
links = sorted(set(own_links) | set(ego_comments['link_id']))

# drop megathreads: they are most of the pull and co-presence in them is not a tie
thread_meta = pd.DataFrame(fetch_by_ids('posts', [l[3:] for l in links],
                                        'id,subreddit,num_comments,author'))
thread_meta['link_id'] = 't3_' + thread_meta['id']
too_big = set(thread_meta.loc[thread_meta['num_comments'] > THREAD_MAX_COMMENTS, 'link_id'])
print(f'threads over {THREAD_MAX_COMMENTS:,} comments dropped: {len(too_big)} of {len(links)} '
      f'({thread_meta.loc[thread_meta.link_id.isin(too_big), "num_comments"].sum():,} comments skipped)')
thread_meta.to_parquet(os.path.join(OUT_DIR, f'thread_sizes_{EGO}.parquet'),
                       engine='pyarrow', compression='zstd')
links = [l for l in links if l not in too_big]
# Split the threads into batches; each batch is cached as one parquet file.
batches = [links[i:i + THREAD_BATCH] for i in range(0, len(links), THREAD_BATCH)]
batch_path = lambda i: os.path.join(CACHE_DIR, f'threads_{i:04d}.parquet')

# a batch file is only valid for the exact thread list it was built from
manifest_path = os.path.join(CACHE_DIR, 'manifest.parquet')
manifest = pd.DataFrame({'link_id': links, 'batch': [i // THREAD_BATCH for i in range(len(links))]})
if os.path.exists(manifest_path) and not pd.read_parquet(manifest_path).equals(manifest):
    raise RuntimeError('thread list changed since the cache was built - clear thread_cache/ first')
manifest.to_parquet(manifest_path, engine='pyarrow')

print(f'threads: {len(links):,}  ({len(set(own_links)):,} own posts, '
      f'{len(links) - len(set(own_links)):,} others)  in {len(batches)} batches')


# Pull every comment in one batch of threads; skip batches already cached; write via a .tmp file so a crash leaves no half file.
def pull_batch(i):
    path = batch_path(i)
    if os.path.exists(path):
        return i, 0
    rows = []
    for link in batches[i]:
        rows.extend(arctic_paginate('comments/search', link_id=link, fields=','.join(COMMENT_FIELDS)))
    df = pd.DataFrame(rows, columns=COMMENT_FIELDS)
    df.to_parquet(path + '.tmp', engine='pyarrow', compression='zstd')
    os.replace(path + '.tmp', path)
    return i, len(df)


# Pull the missing batches in parallel, with a progress line every 5 batches.
pending = [i for i in range(len(batches)) if not os.path.exists(batch_path(i))]
print(f'{len(batches) - len(pending)} batches cached, {len(pending)} to pull with {N_WORKERS} workers')
t_start, n_rows = time.time(), 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = [pool.submit(pull_batch, i) for i in pending]
    for k, fut in enumerate(as_completed(futures), 1):
        _, n = fut.result()
        n_rows += n
        if k % 5 == 0 or k == len(futures):
            print(f'  {k}/{len(futures)} batches  {n_rows:,} comments  '
                  f'{(time.time() - t_start) / 60:.1f} min', flush=True)

In [ ]:
threads = (pd.concat([pd.read_parquet(batch_path(i)) for i in range(len(batches))], ignore_index=True)
             .drop_duplicates('id'))
print(f'thread comments: {len(threads):,}')

# coverage: the ego's own comments should all turn up inside the threads
ego_ids = set(ego_comments['id'])
found = ego_ids & set(threads['id'])
# an ego with no comments of its own (a submitter-only account) is a valid case
if ego_ids:
    print(f'ego comments found inside the thread pull: {len(found):,}/{len(ego_ids):,} '
          f'({len(found) / len(ego_ids):.2%})')
else:
    print('ego wrote no comments: every tie here comes from replies to its submissions')

## 5. Ego ↔ alter interactions

- **ego → alter**: each ego comment's parent author. Parents are looked up in the thread
  pull first; submissions (and any parent comment the thread pull lacks) are resolved via
  `/ids`.
- **alter → ego**: every thread comment whose `parent_id` is ego content.

In [ ]:
# author of every fullname we know about
fullname_author = pd.Series(threads['author'].values, index='t1_' + threads['id'])
fullname_author = pd.concat([fullname_author,
                             pd.Series(EGO, index='t1_' + ego_comments['id']),
                             pd.Series(EGO, index='t3_' + ego_posts['id'])])
fullname_author = fullname_author[~fullname_author.index.duplicated()]

need = ego_comments.loc[~ego_comments['parent_id'].isin(fullname_author.index), 'parent_id']
need_c = need[need.str.startswith('t1_')].str[3:].tolist()
need_p = need[need.str.startswith('t3_')].str[3:].tolist()
print(f'parents to resolve via /ids: {len(need_c):,} comments, {len(need_p):,} submissions')
extra = ([('t1_' + r['id'], r['author']) for r in fetch_by_ids('comments', need_c, 'id,author')]
         + [('t3_' + r['id'], r['author']) for r in fetch_by_ids('posts', need_p, 'id,author')])
if extra:
    fullname_author = pd.concat([fullname_author, pd.Series(dict(extra))])
    fullname_author = fullname_author[~fullname_author.index.duplicated()]

# --- ego -> alter ---
out = ego_comments.assign(alter=ego_comments['parent_id'].map(fullname_author))
n_unresolved = int(out['alter'].isna().sum())
n_out_deleted = int(out['alter'].isin(NON_AUTHORS).sum())
n_out_self = int((out['alter'].str.lower() == EGO.lower()).sum())
out = out[out['alter'].notna() & ~out['alter'].isin(NON_AUTHORS)
          & (out['alter'].str.lower() != EGO.lower())]
out = out.assign(direction='ego_to_alter')

# --- alter -> ego ---
ego_content = set('t1_' + ego_comments['id']) | set('t3_' + ego_posts['id'])
inc = threads[threads['parent_id'].isin(ego_content)]
n_in_deleted = int(inc['author'].isin(NON_AUTHORS).sum())
n_in_self = int((inc['author'].str.lower() == EGO.lower()).sum())
inc = inc[~inc['author'].isin(NON_AUTHORS) & (inc['author'].str.lower() != EGO.lower())]
inc = inc.assign(alter=inc['author'], direction='alter_to_ego')

cols = ['alter', 'direction', 'id', 'parent_id', 'link_id', 'created_utc', 'subreddit']
interactions = pd.concat([out[cols], inc[cols]], ignore_index=True)
interactions['parent_type'] = np.where(interactions['parent_id'].str.startswith('t1_'), 'comment', 'submission')
assert interactions['id'].is_unique, 'a comment counted twice'

print(f'ego -> alter replies: {len(out):,}   (unresolved {n_unresolved:,}, '
      f'[deleted] parent {n_out_deleted:,}, self {n_out_self:,})')
print(f'alter -> ego replies: {len(inc):,}   ([deleted] author {n_in_deleted:,}, self {n_in_self:,})')
print(f'distinct alters (before filtering): {interactions["alter"].nunique():,}')

### 5b. Remove bots

In [ ]:
# Remove bots among the alters: names on the known-bot list (plus EXTRA_BOTS), or matching BOT_NAME_RULE,
# unless listed in NOT_BOTS.
known_bots = {l.strip().lower() for l in open(BOT_LIST_PATH) if l.strip()} | {b.lower() for b in EXTRA_BOTS}
not_bots = {b.lower() for b in NOT_BOTS}


def bot_reason(name):
    n = name.lower()
    if n in not_bots:
        return None
    if n in known_bots:
        return 'known bot'
    if BOT_NAME_RULE.search(name):
        return 'name rule'
    return None


# Tag each alter, list the removed bots, and drop their interactions.
reasons = {a: bot_reason(a) for a in interactions['alter'].unique()}
interactions['bot_reason'] = interactions['alter'].map(reasons)
bots = (interactions[interactions['bot_reason'].notna()]
        .groupby(['alter', 'bot_reason']).agg(posts=('link_id', 'nunique'), replies=('id', 'size'))
        .reset_index().sort_values(['posts', 'alter'], ascending=[False, True]).reset_index(drop=True))
print(f'known bot list: {len(known_bots):,} names')
print(f'bots removed: {len(bots):,} accounts, {bots["replies"].sum():,} replies')
print(bots['bot_reason'].value_counts().to_string())
interactions = interactions[interactions['bot_reason'].isna()].drop(columns='bot_reason').reset_index(drop=True)
print('\nremoved bots (review the "name rule" rows; add false positives to NOT_BOTS):')
bots

## 6. Contacts per alter — counted as distinct posts, and the `MIN_CONTACTS` filter

A **contact** is a distinct post (`link_id`) on which the ego and the alter replied to each
other at least once. A back-and-forth comment chain on one post is still one contact, so
the count measures how many separate conversations the pair shared, not how long any one of
them ran. Reply counts are kept alongside for reference only.

Each shared post is also classed by direction — only the ego replied, only the alter
replied, or both — so the three classes add up exactly to the contact count.

In [ ]:
# Per alter and post: did only the ego reply, only the alter, or both?
per_post = (interactions.groupby(['alter', 'link_id'])['direction']
            .agg(lambda s: 'both' if s.nunique() == 2 else s.iloc[0]))
post_split = (per_post.groupby(level='alter').value_counts().unstack(fill_value=0)
              .reindex(columns=['ego_to_alter', 'alter_to_ego', 'both'], fill_value=0)
              .rename(columns={'ego_to_alter': 'posts_ego_only', 'alter_to_ego': 'posts_alter_only',
                               'both': 'posts_both'}))

# One row per alter: contacts (distinct posts), replies each way, first/last contact, top 3 subreddits.
top_subs = lambda s: ','.join(s.value_counts().head(3).index)
alters_all = (interactions
              .groupby('alter')
              .agg(contacts=('link_id', 'nunique'),
                   replies=('id', 'size'),
                   replies_ego_to_alter=('direction', lambda s: int((s == 'ego_to_alter').sum())),
                   replies_alter_to_ego=('direction', lambda s: int((s == 'alter_to_ego').sum())),
                   first_utc=('created_utc', 'min'),
                   last_utc=('created_utc', 'max'),
                   top_subreddits=('subreddit', top_subs))
              .join(post_split)
              .reset_index())
alters_all['posts_ego_to_alter'] = alters_all['posts_ego_only'] + alters_all['posts_both']
alters_all['posts_alter_to_ego'] = alters_all['posts_alter_only'] + alters_all['posts_both']
alters_all = alters_all.sort_values(['contacts', 'replies', 'alter'],
                                    ascending=[False, False, True]).reset_index(drop=True)

# --- checks on the counting ---
assert (alters_all['posts_ego_only'] + alters_all['posts_alter_only'] + alters_all['posts_both']
        == alters_all['contacts']).all(), 'direction classes do not add up to distinct posts'
assert (alters_all['contacts'] <= alters_all['replies']).all()
assert alters_all['replies'].sum() == len(interactions)
assert alters_all['contacts'].sum() == interactions[['alter', 'link_id']].drop_duplicates().shape[0]
print('PASS  contacts = distinct posts; direction classes add up; no reply lost')

# How much counting posts instead of replies changes things (long reply chains on one post).
chains = alters_all[alters_all['replies'] > alters_all['contacts']]
print(f'\nalters whose replies are concentrated in comment chains (replies > posts): {len(chains):,}')
print(f'replies collapsed by counting posts instead: {int((alters_all["replies"] - alters_all["contacts"]).sum()):,}')
print('largest differences:')
display(alters_all.assign(extra_replies=alters_all['replies'] - alters_all['contacts'])
        .nlargest(10, 'extra_replies')[['alter', 'replies', 'contacts', 'extra_replies']])

# Distribution of alters by number of distinct posts.
dist = alters_all['contacts'].value_counts().sort_index()
print('alters by number of distinct posts:')
for k in [1, 2, 3, 4, 5]:
    print(f'  {k:>3}        : {int(dist.get(k, 0)):>7,}')
print(f'  6-10       : {int(dist[(dist.index >= 6) & (dist.index <= 10)].sum()):>7,}')
print(f'  11+        : {int(dist[dist.index > 10].sum()):>7,}')

# Keep alters with at least MIN_CONTACTS distinct posts.
alters = alters_all[alters_all['contacts'] >= MIN_CONTACTS].reset_index(drop=True)
by_replies = int((alters_all['replies'] >= MIN_CONTACTS).sum())
print(f'\nretained alters (>= {MIN_CONTACTS} distinct posts): {len(alters):,}')
print(f'  (counting replies instead would have retained {by_replies:,} - '
      f'{by_replies - len(alters):,} of those only met the threshold through one comment chain)')

### 6b. Bots identified from reply text

Name rules miss bots with ordinary names (u/example_mirror_bot, u/example_quiet_bot, u/example_relay_bot, u/example_spam_bot
were in the top 30 before this step). For every retained alter the text of its replies is
fetched, and an alter is treated as a bot when at least `BOT_TEXT_SHARE` of its replies
contain bot self-identification ("I'm a bot", "performed automatically", "self-deletes") or a
bot footer linking to its code. Subreddit moderators posting removal notices by hand are *not*
matched and stay in.

In [ ]:
# Second bot screen, by reply text: fetch (and cache) the text of every reply between the ego and a retained alter.
bodies_path = os.path.join(OUT_DIR, 'retained_reply_bodies.parquet')
reply_ids = interactions.loc[interactions['alter'].isin(alters['alter']), 'id'].tolist()
bodies = pd.read_parquet(bodies_path) if os.path.exists(bodies_path) else pd.DataFrame(columns=['id', 'body'])
missing_ids = sorted(set(reply_ids) - set(bodies['id']))
if missing_ids:
    batches_b = [missing_ids[k:k + 100] for k in range(0, len(missing_ids), 100)]
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        fetched = [r for batch in pool.map(lambda b: arctic_get('comments/ids', ids=','.join(b),
                                                                fields='id,body'), batches_b)
                   for r in batch]
    bodies = pd.concat([bodies, pd.DataFrame(fetched, columns=['id', 'body'])], ignore_index=True)
    bodies.to_parquet(bodies_path, engine='pyarrow', compression='zstd')
print(f'reply texts: {bodies["id"].isin(reply_ids).sum():,}/{len(reply_ids):,}')

# Alters whose replies match BOT_TEXT_RULE at least BOT_TEXT_SHARE of the time are treated as bots (unless in NOT_BOTS).
texts = interactions[['id', 'alter']].merge(bodies, on='id')
texts['bot_text'] = texts['body'].fillna('').str.contains(BOT_TEXT_RULE)
text_score = texts.groupby('alter').agg(replies_checked=('id', 'size'), bot_text_share=('bot_text', 'mean'))
text_bots = text_score[(text_score['bot_text_share'] >= BOT_TEXT_SHARE)
                       & ~text_score.index.str.lower().isin(not_bots)]

print(f'bots found from reply text: {len(text_bots)}')
display(alters.set_index('alter').loc[text_bots.index, ['contacts', 'replies']]
        .join(text_bots).sort_values('contacts', ascending=False))
alters = alters[~alters['alter'].isin(text_bots.index)].reset_index(drop=True)
alters_all['bot_text'] = alters_all['alter'].isin(text_bots.index)
print(f'retained alters after text-based bot removal: {len(alters):,}')

## 7. Current account status (PRAW)

Looked up only for retained alters, and cached. Recorded as an attribute; alters are only
removed on it when `DROP_PRAW_DELETED` is set.

In [ ]:
# Current account status (Active / Suspended / Deleted) from PRAW for the ego and retained alters, cached to parquet.
# Credentials come from the .env file.
status_path = os.path.join(OUT_DIR, 'account_status.parquet')
account_status = dict(pd.read_parquet(status_path).values) if os.path.exists(status_path) else {}

if CHECK_ACCOUNT_STATUS:
    assert load_dotenv(ENV_PATH), f'could not load {ENV_PATH}'
    import praw, prawcore
    reddit_api = praw.Reddit(client_id=os.getenv('praw_script'),
                             client_secret=os.getenv('praw_key'),
                             username=os.getenv('login_name'),
                             password=os.getenv('password'),
                             user_agent=os.getenv('user_agent'),
                             ratelimit_seconds=360)
    # Look up only accounts not already cached; retry on rate limits; save the cache every 250 lookups.
    to_check = [n for n in [EGO] + alters['alter'].tolist() if n not in account_status]
    print(f'{len(to_check):,} accounts to look up ({len(account_status):,} cached)')
    t_start = time.time()
    for k, name in enumerate(to_check, 1):
        retry_count = 0
        while retry_count < 10:
            try:
                redditor = reddit_api.redditor(name)
                try:
                    if hasattr(redditor, 'is_suspended'):
                        account_status[name] = 'Suspended'
                    elif hasattr(redditor, 'id'):
                        account_status[name] = 'Active'
                    else:
                        account_status[name] = 'Deleted'
                except prawcore.exceptions.NotFound:
                    account_status[name] = 'Deleted'
                break
            except prawcore.exceptions.TooManyRequests as e:
                retry_count += 1
                print(f'rate limited on {name}, retry {retry_count}')
                if retry_count == 10:
                    raise e
        if k % 250 == 0 or k == len(to_check):
            (pd.DataFrame(list(account_status.items()), columns=['account', 'account_status'])
               .to_parquet(status_path, engine='pyarrow'))
            print(f'  {k:,}/{len(to_check):,}  {(time.time() - t_start) / 60:.1f} min', flush=True)

# Attach status; drop deleted alters only if DROP_PRAW_DELETED is set.
alters['account_status'] = alters['alter'].map(account_status).fillna('not checked')
print(f'ego status: {account_status.get(EGO, "not checked")}')
print(alters['account_status'].value_counts().to_string())

if DROP_PRAW_DELETED:
    before = len(alters)
    alters = alters[alters['account_status'] != 'Deleted'].reset_index(drop=True)
    print(f'removed {before - len(alters):,} PRAW-deleted alters')

## 8. Most frequently contacted alters

In [ ]:
# The 40 most-contacted alters.
ranked = alters.assign(first=pd.to_datetime(alters['first_utc'], unit='s').dt.date,
                       last=pd.to_datetime(alters['last_utc'], unit='s').dt.date)
ranked[['alter', 'contacts', 'posts_ego_only', 'posts_both', 'posts_alter_only', 'replies',
        'account_status', 'first', 'last', 'top_subreddits']].head(40)

## 9. Graph and checks

In [ ]:
# Directed ego network: ego -> alter and alter -> ego edges, weighted by distinct posts with a reply in that direction.
G = nx.DiGraph()
G.add_node(EGO, role='ego', account_status=account_status.get(EGO, 'not checked'))
for _, r in alters.iterrows():
    G.add_node(r['alter'], role='alter', contacts=int(r['contacts']), replies=int(r['replies']),
               account_status=r['account_status'])
    # edge weight = distinct posts with a reply in that direction; replies kept for reference
    if r['posts_ego_to_alter']:
        G.add_edge(EGO, r['alter'], weight=int(r['posts_ego_to_alter']),
                   replies=int(r['replies_ego_to_alter']))
    if r['posts_alter_to_ego']:
        G.add_edge(r['alter'], EGO, weight=int(r['posts_alter_to_ego']),
                   replies=int(r['replies_alter_to_ego']))

checks = {
    'no [deleted]/[removed] nodes': not set(G.nodes()) & NON_AUTHORS,
    'no self-loops': nx.number_of_selfloops(G) == 0,
    f'every alter shares >= {MIN_CONTACTS} distinct posts with the ego': all(
        G.nodes[n]['contacts'] >= MIN_CONTACTS for n in G if n != EGO),
    'every edge touches the ego': all(EGO in e for e in G.edges()),
    'no edge weight exceeds its reply count': all(
        d['weight'] <= d['replies'] for *_, d in G.edges(data=True)),
    'out + in post weights - two-way posts = contacts': all(
        (G[EGO][a]['weight'] if G.has_edge(EGO, a) else 0)
        + (G[a][EGO]['weight'] if G.has_edge(a, EGO) else 0) - r['posts_both'] == r['contacts']
        for a, r in alters.set_index('alter').iterrows()),
    'ego not listed as an alter': EGO.lower() not in set(alters['alter'].str.lower()),
    'no bots among alters': not any(bot_reason(a) for a in alters['alter'])
        and not alters['alter'].isin(text_bots.index).any(),
}
for name, ok in checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}] {name}')
assert all(checks.values())

print(f'\nnodes {G.number_of_nodes():,}  edges {G.number_of_edges():,}')
print(f'alters with at least one two-way post: {int((alters["posts_both"] > 0).sum()):,}')

## 10. Visualisation

**Figure 1 — ego network of the `TOP_N_GRAPH` most-contacted alters.** Closer to the ego =
more distinct shared posts. Orange = the ego replied to the alter, blue = the alter replied
to the ego; width = distinct posts with a reply in that direction.

**Figure 2 — ranked contacts** for the top `TOP_N_BARS` alters: distinct shared posts, split
into posts where only the ego replied, both replied, or only the alter replied.

In [ ]:
EGO_COLOR  = '#0b0b0b'
OUT_COLOR  = '#eb6834'   # ego -> alter     (orange / aqua / blue validated as an adjacent set)
BOTH_COLOR = '#1baf7a'   # both directions  (below 3:1 on the surface: bars carry value labels)
IN_COLOR   = '#2a78d6'   # alter -> ego
MUTED      = '#52514e'
SURFACE    = '#fcfcfb'
# account-outcome colours taken from Figures/Reddit_figs/BlackPeopleTwitterclusters1.svg
STATUS_COLORS = {'Active': '#4b2362',      # dark purple
                 'Suspended': '#eeb182',   # tan
                 'Deleted': '#cfccc4',     # neutral grey (not in the source figure)
                 'not checked': '#ffffff'}


def draw_top_ego_network(G, alters, ego, top_n, figsize=(13, 13)):
    top = alters.head(top_n).reset_index(drop=True)
    cmin, cmax = top['contacts'].min(), top['contacts'].max()
    # golden-angle spiral: rank 0 nearest the ego, successive ranks spread round the circle
    golden = np.pi * (3 - np.sqrt(5))
    pos = {ego: np.array([0.0, 0.0])}
    for i, r in top.iterrows():
        closeness = 0.0 if cmax == cmin else (np.log(r['contacts']) - np.log(cmin)) / (np.log(cmax) - np.log(cmin))
        radius = 0.3 + 0.7 * (1 - closeness)
        a = i * golden
        pos[r['alter']] = radius * np.array([np.cos(a), np.sin(a)])

    drawn = [G[u][v]['weight'] for u, v in G.edges() if u in pos and v in pos]
    wmin, wmax = min(drawn), max(drawn)
    width = lambda w: 1.2 if wmax == wmin else 0.7 + 3.3 * (w - wmin) / (wmax - wmin)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for _, r in top.iterrows():
        a = r['alter']
        both = G.has_edge(ego, a) and G.has_edge(a, ego)
        for u, v, color in ((ego, a, OUT_COLOR), (a, ego, IN_COLOR)):
            rad = 0.15 if both else 0.0      # curve only to separate a two-way pair
            if G.has_edge(u, v):
                ax.annotate('', xy=pos[v], xytext=pos[u], zorder=2,
                            arrowprops=dict(arrowstyle='-|>', mutation_scale=9,
                                            lw=width(G[u][v]['weight']), color=color, alpha=0.75,
                                            shrinkA=8, shrinkB=8, connectionstyle=f'arc3,rad={rad}'))
    sizes = 40 + 260 * (top['contacts'] - cmin) / max(cmax - cmin, 1)
    # fill = current account outcome; the dark outline keeps the light fills legible
    ax.scatter([pos[a][0] for a in top['alter']], [pos[a][1] for a in top['alter']],
               s=sizes, c=[STATUS_COLORS.get(st, '#ffffff') for st in top['account_status']],
               edgecolors=MUTED, linewidths=0.9, zorder=3)
    ax.scatter([0], [0], s=600, c=EGO_COLOR, edgecolors=SURFACE, linewidths=2, zorder=4)
    ax.annotate(f'u/{ego}', (0, 0), xytext=(0, -15), textcoords='offset points', ha='center',
                va='top', fontsize=9, fontweight='bold', color=EGO_COLOR, zorder=6,
                bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.9))
    for _, r in top.iterrows():
        ax.annotate(f'{r["alter"]} ({r["contacts"]})', pos[r['alter']], xytext=(0, 8),
                    textcoords='offset points', ha='center', va='bottom', fontsize=7,
                    color='#0b0b0b', zorder=6,
                    bbox=dict(boxstyle='round,pad=0.12', fc=SURFACE, ec='none', alpha=0.8))

    counts = top['account_status'].value_counts()
    handles = [Line2D([], [], color=OUT_COLOR, lw=2, label=f'u/{ego} replied to alter'),
               Line2D([], [], color=IN_COLOR, lw=2, label=f'alter replied to u/{ego}')]
    handles += [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=0.9,
                       markersize=9, label=f'{st} ({counts[st]})')
                for st in ('Active', 'Suspended', 'Deleted', 'not checked') if counts.get(st)]
    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8.5,
              title='reply direction / account outcome', title_fontsize=8.5,
              alignment='left')
    fig.text(0.02, 0.975, f'Most-contacted alters of u/{ego}', fontsize=13, color='#0b0b0b', va='top')
    fig.text(0.02, 0.949,
             f'top {len(top)} of {len(alters):,} alters sharing ≥{MIN_CONTACTS} posts with the ego · '
             f'contact = distinct post, not reply · [deleted], bot and one-post alters removed\n'
             f'closer to the ego = more shared posts · width = posts per direction · '
             f'label = shared posts · node fill = current account outcome',
             fontsize=8.5, color=MUTED, va='top', linespacing=1.5)
    ax.set_xlim(-1.18, 1.18); ax.set_ylim(-1.18, 1.18)
    ax.set_aspect('equal'); ax.axis('off')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02)
    return fig, ax


# an ego whose alters all share a single post has nothing to draw here
if len(alters):
    fig, ax = draw_top_ego_network(G, alters, EGO, TOP_N_GRAPH)
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_top{TOP_N_GRAPH}.png'), dpi=150, facecolor=SURFACE)
    plt.show()

In [ ]:
# Figure: horizontal bars of the top alters' contacts, split by who replied (ego only, both, alter only).
def draw_ranked_contacts(alters, ego, top_n):
    top = alters.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 0.28 * len(top) + 1.6))
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    y = np.arange(len(top))
    left = np.zeros(len(top))
    for col, color, label in (('posts_ego_only', OUT_COLOR, f'only u/{ego} replied'),
                              ('posts_both', BOTH_COLOR, 'both replied'),
                              ('posts_alter_only', IN_COLOR, f'only the alter replied')):
        ax.barh(y, top[col], left=left, height=0.7, color=color, edgecolor=SURFACE, linewidth=1.5,
                label=label)
        left += top[col].to_numpy()
    for yi, total, replies in zip(y, top['contacts'], top['replies']):
        ax.text(total + top['contacts'].max() * 0.01, yi, f'{total}  ({replies} replies)',
                va='center', fontsize=7.5, color=MUTED)
    ax.set_yticks(y, top['alter'], fontsize=8)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', colors=MUTED, labelsize=8)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color('#d6d4cf')
    ax.xaxis.grid(True, color='#e8e6e1', lw=0.6); ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlim(0, top['contacts'].max() * 1.22)
    ax.set_xlabel('distinct posts with a direct reply between the pair', fontsize=8.5, color=MUTED)
    ax.legend(loc='lower right', frameon=False, fontsize=8.5)
    ax.set_title(f'Top {len(top)} alters of u/{ego} by shared posts', loc='left', fontsize=12, color='#0b0b0b')
    fig.tight_layout()
    return fig, ax


# an ego whose alters all share a single post has nothing to draw here
if len(alters):
    fig, ax = draw_ranked_contacts(alters, EGO, TOP_N_BARS)
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_ranked_contacts.png'), dpi=150, facecolor=SURFACE)
    plt.show()

## 11. Alter–alter ties **within the ego's threads**

Ties between the retained alters, built from the thread pull that is already cached — no
extra collection. A tie exists where one retained alter replied directly to another in a
thread u/CaseA posted or commented in.

**This is a narrower definition than the u/CaseA notebook uses.** There, every
alter's full post history was pulled, so alter–alter replies anywhere on Reddit are counted.
Doing that here would mean ~250M items (≈25–35k alters × ~8.3k items each), days of
requests, so ties are counted only in the ego's own conversational context. Any tie found
here is real; ties the pair formed elsewhere are not counted.

Ties are **undirected** and weighted by distinct shared posts, as in the ego–alter table.

In [ ]:
# Alter-alter ties inside the ego's threads (no new collection): a tie is a retained alter replying to another
# retained alter; weight = distinct posts with such a reply.
MIN_TIE_POSTS = 2          # ties drawn in the figure; the saved table keeps all of them

threads_all = (pd.concat([pd.read_parquet(batch_path(i)) for i in range(len(batches))],
                         ignore_index=True).drop_duplicates('id'))

# parent authors: every comment in the pull, plus the submission each thread hangs off
post_author_path = os.path.join(OUT_DIR, 'thread_post_authors.parquet')
if os.path.exists(post_author_path):
    thread_posts = pd.read_parquet(post_author_path)
else:
    ids = [l[3:] for l in sorted(threads_all['link_id'].unique())]
    thread_posts = pd.DataFrame(fetch_by_ids('posts', ids, 'id,author'), columns=['id', 'author'])
    thread_posts.to_parquet(post_author_path, engine='pyarrow', compression='zstd')

parent_author = pd.concat([
    pd.Series(threads_all['author'].values, index='t1_' + threads_all['id']),
    pd.Series(thread_posts['author'].values, index='t3_' + thread_posts['id']),
])
parent_author = parent_author[~parent_author.index.duplicated()]

kept = set(alters['alter'])
tie_rows = threads_all.assign(parent_author=threads_all['parent_id'].map(parent_author))
tie_rows = tie_rows[tie_rows['author'].isin(kept) & tie_rows['parent_author'].isin(kept)
                    & (tie_rows['author'] != tie_rows['parent_author'])]

u = np.minimum(tie_rows['author'], tie_rows['parent_author'])
v = np.maximum(tie_rows['author'], tie_rows['parent_author'])
alter_ties = (tie_rows.assign(a=u, b=v)
              .groupby(['a', 'b'])
              .agg(posts=('link_id', 'nunique'), replies=('id', 'size'),
                   first_utc=('created_utc', 'min'), last_utc=('created_utc', 'max'),
                   subreddits=('subreddit', lambda s: ','.join(sorted(set(s))[:3])))
              .reset_index()
              .sort_values(['posts', 'replies'], ascending=False)
              .reset_index(drop=True))

assert (alter_ties['posts'] <= alter_ties['replies']).all()
assert alter_ties['replies'].sum() == len(tie_rows)
assert not alter_ties.duplicated(subset=['a', 'b']).any()
assert not (alter_ties['a'] == alter_ties['b']).any()
assert set(alter_ties['a']) | set(alter_ties['b']) <= kept, 'a tie endpoint is not a retained alter'
print('PASS  ties are undirected, one row per pair, both endpoints retained alters')

AT = nx.Graph()
AT.add_nodes_from(kept)
for _, r in alter_ties.iterrows():
    AT.add_edge(r['a'], r['b'], weight=int(r['posts']), n_replies=int(r['replies']))
tied_alters = [n for n in AT.nodes() if AT.degree(n) > 0]

print(f'alter -> alter replies in the ego\'s threads: {len(tie_rows):,}')
print(f'undirected ties: {len(alter_ties):,}  ({int(alter_ties["posts"].sum()):,} shared posts)')
print(f'alters with >=1 alter tie: {len(tied_alters):,} of {len(kept):,}'
      + (f' ({len(tied_alters) / len(kept):.1%})' if kept else '  (no alter cleared MIN_CONTACTS)'))
HAS_TIES = len(alter_ties) > 0
if not HAS_TIES:
    print('\nNo alter-alter ties: none of the retained alters ever replied to each other in one of '
          f"u/{EGO}'s threads. Sections 11.1-11.2 below are skipped.")
if HAS_TIES:
    print(f'tie weight range: {int(alter_ties["posts"].min())} - {int(alter_ties["posts"].max())}')
print()
print('how a weight threshold thins the network:')
for thr in (1, 2, 3, 4) if HAS_TIES else ():
    sub = alter_ties[alter_ties['posts'] >= thr]
    H = nx.Graph(); H.add_edges_from(sub[['a', 'b']].itertuples(index=False, name=None))
    comps = sorted(nx.connected_components(H), key=len, reverse=True) if H.number_of_nodes() else []
    print(f'  >= {thr} shared post(s): {len(sub):>6,} ties  {H.number_of_nodes():>5,} alters  '
          f'main group {len(comps[0]) if comps else 0:>5,}')
alter_ties.head(15)

### 11.1 Figure — alter–alter ties, main connected group

Same construction as the u/CaseA figure: undirected ties, width = distinct
shared posts, node fill = current account outcome, ego drawn in with faint ties. Only ties of
at least `MIN_TIE_POSTS` shared posts are drawn, because every tie (1,383 alters, one big
group) is an unreadable hairball; the saved table and graph keep them all.

In [ ]:
# Figure helpers and the alter-alter tie figure (main connected group; width = shared posts, fill = account status).
TIE_EDGE = '#7a7873'
EGO_EDGE = '#b5b3ad'


def label_offset(ax, x, y, text, **kw):
    return ax.annotate(text, (x, y), xytext=(0, 6), textcoords='offset points',
                       ha='center', va='bottom', fontsize=7, color='#0b0b0b', zorder=6,
                       bbox=dict(boxstyle='round,pad=0.12', fc=SURFACE, ec='none', alpha=0.75), **kw)


def status_handles(names, account_status, marker_size=9):
    counts = pd.Series([account_status.get(n, 'not checked') for n in names]).value_counts()
    return [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=0.9,
                   markersize=marker_size, label=f'{st} ({counts[st]})')
            for st in ('Active', 'Suspended', 'Deleted', 'not checked') if counts.get(st)]


def place_ego(pos, graph, ego, n_grid=31, search=0.3, min_clear=0.05, pass_radius=0.03):
    # closest spot to the middle of the alter layout that no alter-alter tie passes through
    xy = np.array([pos[n] for n in pos])
    centre = xy.mean(axis=0)
    span = np.ptp(xy, axis=0).max()
    P = np.array([pos[a] for a, b in graph.edges()])
    Q = np.array([pos[b] for a, b in graph.edges()])
    D = Q - P
    DD = np.maximum((D ** 2).sum(axis=1), 1e-12)
    best = None
    for gx in np.linspace(-search, search, n_grid):
        for gy in np.linspace(-search, search, n_grid):
            p = centre + span * np.array([gx, gy])
            t = np.clip(((p - P) * D).sum(axis=1) / DD, 0, 1)
            crossings = int((np.linalg.norm(P + t[:, None] * D - p, axis=1) < pass_radius * span).sum())
            clearance = np.linalg.norm(xy - p, axis=1).min() / span
            key = (crossings, clearance < min_clear, np.hypot(gx, gy))
            if best is None or key < best[0]:
                best = (key, p)
    return best[1]


def draw_alter_ties(alter_ties, account_status, ego, min_posts=MIN_TIE_POSTS,
                    max_labels=25, figsize=(14, 14)):
    sub = alter_ties[alter_ties['posts'] >= min_posts]
    H = nx.Graph()
    H.add_edges_from((r['a'], r['b'], {'weight': int(r['posts'])}) for _, r in sub.iterrows())
    if H.number_of_edges() == 0:
        print('no ties at this threshold')
        return None, None
    comps = sorted(nx.connected_components(H), key=len, reverse=True)
    S = nx.Graph()                                   # sorted order keeps the layout reproducible
    S.add_nodes_from(sorted(comps[0]))
    S.add_edges_from((a, b, d) for a, b, d in H.subgraph(comps[0]).edges(data=True))
    left_out = comps[1:]

    pos = nx.kamada_kawai_layout(S)
    pos[ego] = place_ego(pos, S, ego)
    deg = dict(S.degree())
    wmax = max(d['weight'] for _, _, d in S.edges(data=True))
    dmax = max(deg.values())
    sizes = {n: 35 + 240 * np.sqrt(deg[n] / dmax) for n in S.nodes()}   # sqrt keeps hubs from ballooning

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for n in S.nodes():                              # faint ego ties, drawn underneath
        ax.plot([pos[ego][0], pos[n][0]], [pos[ego][1], pos[n][1]], color=EGO_EDGE,
                lw=0.5, alpha=0.35, solid_capstyle='round', zorder=1)
    nx.draw_networkx_edges(S, pos, ax=ax, edge_color=TIE_EDGE, alpha=0.85,
                           width=[0.8 + 2.8 * (d['weight'] - min_posts) / max(wmax - min_posts, 1)
                                  for _, _, d in S.edges(data=True)])
    nx.draw_networkx_nodes(S, pos, ax=ax,
                           node_color=[STATUS_COLORS[account_status.get(n, 'not checked')] for n in S.nodes()],
                           node_size=[sizes[n] for n in S.nodes()],
                           edgecolors=MUTED, linewidths=0.8)
    ax.scatter([pos[ego][0]], [pos[ego][1]], s=620, c=EGO_COLOR, edgecolors=SURFACE,
               linewidths=2.5, zorder=5)
    labels = [label_offset(ax, *pos[n], n)
              for n in sorted(S.nodes(), key=lambda n: (-deg[n], n.lower()))[:max_labels]]

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [t.get_window_extent(renderer) for t in labels]
    node_px = ax.transData.transform(np.array([pos[n] for n in S.nodes()]))
    best = None
    for offset, ha, va in [((0, -18), 'center', 'top'), ((0, 18), 'center', 'bottom'),
                           ((-20, 0), 'right', 'center'), ((20, 0), 'left', 'center'),
                           ((-16, -16), 'right', 'top'), ((16, -16), 'left', 'top'),
                           ((-16, 16), 'right', 'bottom'), ((16, 16), 'left', 'bottom')]:
        t = ax.annotate(f'u/{ego}', pos[ego], xytext=offset, textcoords='offset points', ha=ha,
                        va=va, fontsize=9.5, fontweight='bold', color=EGO_COLOR, zorder=8,
                        bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.95))
        bb = t.get_window_extent(renderer).expanded(1.1, 1.3)
        score = (sum(bb.overlaps(b) for b in boxes), sum(bb.contains(x, y) for x, y in node_px))
        if best is None or score < best[0]:
            if best is not None:
                best[1].remove()
            best = (score, t)
        else:
            t.remove()

    handles = [Line2D([], [], marker='o', ls='', color=EGO_COLOR, markersize=10, label=f'ego (u/{ego})'),
               Line2D([], [], color=TIE_EDGE, lw=2, label=f'alter - alter tie ({S.number_of_edges()})'),
               Line2D([], [], color=EGO_EDGE, lw=2, label=f'ego - alter tie ({S.number_of_nodes()})')]
    handles += status_handles(S.nodes(), account_status)
    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8.5,
              title='ties / account outcome', title_fontsize=8.5, alignment='left')
    fig.text(0.02, 0.975, f'Ties among the alters of u/{ego} — main connected group',
             fontsize=13, color='#0b0b0b', va='top')
    fig.text(0.02, 0.947,
             f'{S.number_of_nodes():,} alters · {S.number_of_edges():,} ties of ≥{min_posts} shared posts · '
             f'{len(left_out)} smaller group(s) of {sum(map(len, left_out))} alters not shown · undirected\n'
             f'ties counted only in threads u/{ego} took part in · width = distinct shared posts · '
             f'node size = tie count · node fill = account outcome · {len(labels)} most-connected labelled',
             fontsize=8.5, color=MUTED, va='top', linespacing=1.5)
    ax.set_aspect('equal'); ax.axis('off'); ax.margins(0.05)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02)
    return fig, ax


if HAS_TIES:
    fig, ax = draw_alter_ties(alter_ties, account_status, EGO)
    if fig is not None:
        fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_alter_ties.png'), dpi=150, facecolor=SURFACE)
        plt.show()

In [ ]:
# Save the alter-alter ties (table and GraphML) and each alter's tie counts next to its ego contact counts.
if HAS_TIES:
    alter_ties.to_parquet(os.path.join(OUT_DIR, f'alter_ties_{EGO}.parquet'),
                          engine='pyarrow', compression='zstd')
    nx.write_graphml(AT.subgraph(tied_alters), os.path.join(OUT_DIR, f'alter_ties_{EGO}.graphml'))

    # tie counts per alter, alongside the ego contact counts
    tie_degree = pd.DataFrame({
        'alter': sorted(kept),
        'alter_ties': [AT.degree(n) for n in sorted(kept)],
        'alter_shared_posts': [AT.degree(n, weight='weight') for n in sorted(kept)],
    }).merge(alters[['alter', 'contacts', 'replies', 'account_status']], on='alter')
    tie_degree = tie_degree.sort_values(['alter_ties', 'alter_shared_posts'],
                                        ascending=False).reset_index(drop=True)
    tie_degree.to_parquet(os.path.join(OUT_DIR, f'alter_tie_degree_{EGO}.parquet'),
                          engine='pyarrow', compression='zstd')
    print('most connected alters (ties within the ego\'s threads):')
    display(tie_degree.head(15))
    print(tie_degree.groupby('account_status')['alter_ties'].describe()[['count', 'mean', 'max']].to_string())

### 11.2 The densest part of the alter–alter network

"Densest" in the standard sense: the subgraph with the highest **average degree**
(`edges / nodes`), found by greedy peeling — repeatedly drop the least-connected alter and
keep the best subgraph seen (Charikar's algorithm, a 2-approximation of the exact optimum).
It is reported next to the **degeneracy core** (the deepest k-core) for comparison.

The ego is drawn in but excluded from the search: u/CaseA is tied to every alter by
construction, so including it would only inflate the density of whatever it joined.

The map places alters on rings by **k-core shell** (deepest core innermost) rather than by
force layout. At this density a force layout is a featureless ball, and the community
structure is weak (modularity ≈ 0.17, only ~50% of ties inside communities, every one of
them r/The_Donald), so there are no separable groups to lay out — it is one cohesive core.

In [ ]:
# Densest part of the alter-alter network (highest average degree, greedy peeling) and its k-core shells.
if HAS_TIES:
    def densest_subgraph(graph):
        # Charikar greedy peeling: drop the lowest-degree node repeatedly and keep the best
        # average-degree subgraph seen. Returns (nodes, edges/nodes).
        H = graph.copy()
        best = (list(H.nodes()), H.number_of_edges() / max(H.number_of_nodes(), 1))
        while H.number_of_nodes() > 2:
            n = min(H.degree(), key=lambda kv: kv[1])[0]
            H.remove_node(n)
            d = H.number_of_edges() / H.number_of_nodes()
            if d > best[1]:
                best = (list(H.nodes()), d)
        return best


    tie_graph = AT.subgraph(tied_alters)
    dense_nodes, dense_density = densest_subgraph(tie_graph)
    D = tie_graph.subgraph(sorted(dense_nodes)).copy()
    core_num = nx.core_number(D)
    degeneracy = max(nx.core_number(tie_graph).values())
    deep_core = nx.k_core(tie_graph, degeneracy)

    assert D.number_of_edges() / D.number_of_nodes() >= tie_graph.number_of_edges() / tie_graph.number_of_nodes()
    assert set(D.nodes()) <= set(alters['alter']), 'densest subgraph contains a non-alter'
    assert EGO not in D, 'ego must not be part of the density search'
    print(f'whole tie network : {tie_graph.number_of_nodes():,} alters, {tie_graph.number_of_edges():,} ties, '
          f'avg degree {2 * tie_graph.number_of_edges() / tie_graph.number_of_nodes():.2f}, '
          f'density {nx.density(tie_graph):.4f}')
    print(f'densest subgraph  : {D.number_of_nodes():,} alters, {D.number_of_edges():,} ties, '
          f'avg degree {2 * D.number_of_edges() / D.number_of_nodes():.2f}, density {nx.density(D):.3f}')
    print(f'degeneracy core   : k={degeneracy}, {deep_core.number_of_nodes():,} alters, '
          f'{deep_core.number_of_edges():,} ties, density {nx.density(deep_core):.3f} '
          f'({len(set(deep_core) & set(D)):,} of its alters are inside the densest subgraph)')
    print(f'k-core shells in the densest subgraph: '
          f'{dict(sorted(pd.Series(core_num).value_counts().items()))}')
    print(f'diameter {nx.diameter(D)} · average clustering {nx.average_clustering(D):.3f} · '
          f'degree assortativity {nx.degree_assortativity_coefficient(D):+.3f}')

    dense_members = (pd.DataFrame({'alter': sorted(D.nodes()),
                                   'ties_in_core': [D.degree(n) for n in sorted(D.nodes())],
                                   'core_shell': [core_num[n] for n in sorted(D.nodes())]})
                     .merge(tie_degree[['alter', 'alter_ties', 'contacts', 'replies', 'account_status']],
                            on='alter')
                     .sort_values(['ties_in_core', 'alter_ties'], ascending=False)
                     .reset_index(drop=True))
    print()
    print('account outcome inside the densest subgraph vs all retained alters:')
    print(pd.DataFrame({'densest subgraph': dense_members['account_status'].value_counts(),
                        'all retained': alters['account_status'].value_counts()}).fillna(0).astype(int).to_string())
    print(f"\nmean ego contacts: {dense_members['contacts'].mean():.2f} inside the densest subgraph "
          f"vs {alters['contacts'].mean():.2f} across all retained alters")
    dense_members.head(20)

In [ ]:
# Figure: the densest subgraph, drawn as k-core rings (deepest core innermost), fill = account status.
def draw_dense_core(D, core_num, account_status, ego, max_labels=20, figsize=(14, 14)):
    shells = sorted(set(core_num.values()), reverse=True)     # deepest core innermost
    radii = np.linspace(0.42, 1.0, len(shells)) if len(shells) > 1 else [0.7]
    deg = dict(D.degree())
    pos = {ego: np.array([0.0, 0.0])}
    for radius, shell in zip(radii, shells):
        members = sorted([n for n in D.nodes() if core_num[n] == shell], key=lambda n: -deg[n])
        # deal the degree-ordered members round the ring with a coprime stride, so the
        # best-connected ones (the labelled ones) end up spread out instead of adjacent
        n_m = len(members)
        stride = max(int(round(n_m * 0.382)), 1)
        while n_m > 1 and np.gcd(stride, n_m) != 1:
            stride += 1
        for i, n in enumerate(members):
            slot = (i * stride) % n_m
            ang = 2 * np.pi * slot / n_m + np.pi / 2
            r = radius if slot % 2 == 0 else radius * 0.94   # stagger so neighbours do not touch
            pos[n] = np.array([r * np.cos(ang), r * np.sin(ang)])

    dmax = max(deg.values())
    sizes = {n: 30 + 200 * np.sqrt(deg[n] / dmax) for n in D.nodes()}
    wmax = max(d['weight'] for _, _, d in D.edges(data=True))

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for radius, shell in zip(radii, shells):
        ax.add_patch(plt.Circle((0, 0), radius, fill=False, ec='#e6e4df', lw=0.8, zorder=0))
        ax.annotate(f'{shell}-core', (0, radius), xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7.5, color=MUTED, zorder=1)
    for n in D.nodes():                                       # ego ties, faint
        ax.plot([0, pos[n][0]], [0, pos[n][1]], color=EGO_EDGE, lw=0.35, alpha=0.2,
                solid_capstyle='round', zorder=1)
    for u, v, d in D.edges(data=True):
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=TIE_EDGE,
                lw=0.4 + 1.8 * (d['weight'] - 1) / max(wmax - 1, 1),
                alpha=0.18 + 0.32 * (d['weight'] - 1) / max(wmax - 1, 1),
                solid_capstyle='round', zorder=2)
    ax.scatter([pos[n][0] for n in D.nodes()], [pos[n][1] for n in D.nodes()],
               s=[sizes[n] for n in D.nodes()],
               c=[STATUS_COLORS[account_status.get(n, 'not checked')] for n in D.nodes()],
               edgecolors=MUTED, linewidths=0.7, zorder=3)
    ax.scatter([0], [0], s=620, c=EGO_COLOR, edgecolors=SURFACE, linewidths=2.5, zorder=4)
    ax.annotate(f'u/{ego}', (0, 0), xytext=(0, -15), textcoords='offset points', ha='center',
                va='top', fontsize=9.5, fontweight='bold', color=EGO_COLOR, zorder=6,
                bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.95))
    for n in sorted(D.nodes(), key=lambda n: -deg[n])[:max_labels]:
        x, y = pos[n]
        ang = np.arctan2(y, x)
        ax.annotate(f'{n} ({deg[n]})', (x, y),
                    xytext=(14 * np.cos(ang), 14 * np.sin(ang)), textcoords='offset points',
                    ha='left' if np.cos(ang) >= 0 else 'right', va='center', fontsize=7,
                    color='#0b0b0b', zorder=6,
                    bbox=dict(boxstyle='round,pad=0.12', fc=SURFACE, ec='none', alpha=0.85))

    handles = [Line2D([], [], marker='o', ls='', color=EGO_COLOR, markersize=10, label=f'ego (u/{ego})'),
               Line2D([], [], color=TIE_EDGE, lw=2, label=f'alter - alter tie ({D.number_of_edges():,})'),
               Line2D([], [], color=EGO_EDGE, lw=2, label=f'ego - alter tie ({D.number_of_nodes():,})')]
    handles += status_handles(D.nodes(), account_status)
    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8.5,
              title='ties / account outcome', title_fontsize=8.5, alignment='left')
    fig.text(0.02, 0.975, f'Densest part of the alter network of u/{ego}',
             fontsize=13, color='#0b0b0b', va='top')
    fig.text(0.02, 0.947,
             f'{D.number_of_nodes():,} alters · {D.number_of_edges():,} undirected ties · '
             f'average degree {2 * D.number_of_edges() / D.number_of_nodes():.1f} · density {nx.density(D):.3f} · '
             f'rings = k-core shell, deepest innermost\n'
             f'ties counted only in threads u/{ego} took part in · width = distinct shared posts · '
             f'node size = ties inside this subgraph · node fill = account outcome · '
             f'{max_labels} most-connected labelled',
             fontsize=8.5, color=MUTED, va='top', linespacing=1.5)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal'); ax.axis('off')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02)
    return fig, ax


if HAS_TIES:
    fig, ax = draw_dense_core(D, core_num, account_status, EGO)
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_dense_core.png'), dpi=150, facecolor=SURFACE)
    plt.show()

    dense_members.to_parquet(os.path.join(OUT_DIR, f'dense_core_{EGO}.parquet'),
                             engine='pyarrow', compression='zstd')
    nx.write_graphml(D, os.path.join(OUT_DIR, f'dense_core_{EGO}.graphml'))

## 12. Save outputs

In [ ]:
# Save interactions, all alters, retained alters and the graph; check the GraphML reloads identically; list the outputs.
interactions.to_parquet(os.path.join(OUT_DIR, f'interactions_{EGO}.parquet'), engine='pyarrow', compression='zstd')
alters_all.to_parquet(os.path.join(OUT_DIR, f'alters_all_{EGO}.parquet'), engine='pyarrow', compression='zstd')
alters.to_parquet(os.path.join(OUT_DIR, f'alters_min{MIN_CONTACTS}_{EGO}.parquet'), engine='pyarrow', compression='zstd')
nx.write_graphml(G, os.path.join(OUT_DIR, f'ego_network_{EGO}_min{MIN_CONTACTS}.graphml'))

rb = nx.read_graphml(os.path.join(OUT_DIR, f'ego_network_{EGO}_min{MIN_CONTACTS}.graphml'))
assert rb.number_of_nodes() == G.number_of_nodes() and rb.number_of_edges() == G.number_of_edges()
print('PASS  round trip verified')
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p):
        print(f'  {p}  ({os.path.getsize(p) / 1024:.1f} KB)')